# 🌌 POLYDIM-EINSOF: Laboratorio QLoRA para Alumnos a Costo $0 USD
**Estándar Pedagógico 2030 | Ariel García Traba**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGT1973/polydim-suite/blob/main/einsof_student_hub/colab_zero_dollar_qlora.ipynb)

Este notebook permite a cualquier alumno o investigador probar los **modelos y adaptadores soberanos QLoRA 2027** en una GPU Tesla T4 gratuita de Google Colab (**sin pagar servidores, sin suscripciones y sin consumir tokens de pago**).

In [ ]:
# 1. INSTALACIÓN DE DEPENDENCIAS EN COLAB (GPU T4 GRATUITA)
!pip install -q -U torch transformers peft bitsandbytes accelerate huggingface_hub

import torch
print(f"PyTorch: {torch.__version__} | GPU Activa: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Sin GPU'}")
assert torch.cuda.is_available(), "Por favor activá la GPU en Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU"

In [ ]:
# 2. SELECCIÓN DEL MODELO Y ADAPTADOR EINSOF A PROBAR
# Los pesos se descargan GRATIS desde Hugging Face (Hugging Face no cobra por descargar modelos abiertos)

CATALOGO = {
    "1": {
        "nombre": "DeepSeek Prover V2 7B (Matemáticas formales y Lean 4)",
        "adapter": "Polydim-Einsof/einsof2-deepseek-prover-v2-stiefel-adapter",
        "base": "deepseek-ai/DeepSeek-Prover-V2-7B"
    },
    "2": {
        "nombre": "GLM-4 9B Chat (Razonamiento bilingüe y general)",
        "adapter": "Polydim-Einsof/einsof2-glm4-9b-stiefel-adapter",
        "base": "THUDM/glm-4-9b-chat"
    },
    "3": {
        "nombre": "Qwen 2.5 1.5B Dual T4 (Ultra-liviano y veloz)",
        "adapter": "Polydim-Einsof/einsof2-qlora-dual-t4-adapter",
        "base": "Qwen/Qwen2.5-1.5B-Instruct"
    }
}

# Elegí el modelo cambiando el número:
ELECCION = "1"
SELECCIONADO = CATALOGO[ELECCION]
print(f"Cargando: {SELECCIONADO['nombre']}")

In [ ]:
# 3. CARGA DEL MODELO BASE CON CUANTIZACIÓN 4-BIT NF4
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(SELECCIONADO["adapter"], trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    SELECCIONADO["base"],
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# Inyectamos el adaptador LoRA retractado en float32 con defecto <= 1e-6
model = PeftModel.from_pretrained(base_model, SELECCIONADO["adapter"])
model.eval()
print("✅ Modelo y adaptador QLoRA 2027 cargados exitosamente en VRAM!")

In [ ]:
# 4. INFERENCIA CON COMPUERTA DE VERDAD 1-TOKEN <|cag_truth|>
pregunta_alumno = "Demuestra que en la variedad Stiefel St(k, D), el defecto de isometría ||W^T W - I||_F es invariante ante rotaciones ortogonales."

mensajes = [{"role": "user", "content": pregunta_alumno}]
prompt_tokens = tokenizer.apply_chat_template(mensajes, return_tensors="pt", add_generation_prompt=True).to("cuda")

with torch.no_grad():
    salida = model.generate(
        prompt_tokens,
        max_new_tokens=400,
        do_sample=False, # Determinismo radical (Regla 1 Gandalf)
        pad_token_id=tokenizer.eos_token_id
    )

texto_generado = tokenizer.decode(salida[0][prompt_tokens.shape[1]:], skip_special_tokens=True)
print("=== RESPUESTA DEL MODELO SOBERANO ===\n")
print(texto_generado)

## 🕊️ Protocolo Helena: Envío de Tensores sin Gastar Tokens
Si no tenés GPU y querés analizar tensores en ChatGPT, Claude o Gemini gratuitos, podés empaquetar el estado en una imagen PNG 128x64.

In [ ]:
# 5. GENERADOR DE PORTADOR HELENA PNG (COSTO $0 TOKENS)
import numpy as np, zlib, struct, hashlib

def crear_portador_helena(vector: np.ndarray, nombre_archivo: str = "helena_carrier.png"):
    vec = np.asarray(vector, dtype=np.float32).ravel()
    padded = np.zeros(128 * 64, dtype=np.float32)
    padded[:len(vec)] = vec
    raw_lines = bytearray()
    for y in range(64):
        raw_lines.append(0)
        raw_lines.extend(padded[y * 128 : (y + 1) * 128].tobytes())
    compressed = zlib.compress(bytes(raw_lines), level=9)
    seal = hashlib.blake2s(vec.tobytes()).hexdigest()[:16]
    
    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data) & 0xffffffff)
    
    png = (b"\x89PNG\r\n\x1a\n" + 
           chunk(b"IHDR", struct.pack(">IIBBBBB", 128, 64, 8, 6, 0, 0, 0)) +
           chunk(b"tEXt", b"Helena\x00D=" + str(len(vec)).encode() + b"|SEAL=" + seal.encode()) +
           chunk(b"IDAT", compressed) +
           chunk(b"IEND", b""))
    
    with open(nombre_archivo, "wb") as f:
        f.write(png)
    print(f"✅ Portador Helena generado: {nombre_archivo} ({len(png)} bytes). Listo para adjuntar en cualquier chat de IA!")

# Prueba de vector normalizado en S^511
mi_vector = np.random.randn(512).astype(np.float32)
mi_vector /= np.linalg.norm(mi_vector)
crear_portador_helena(mi_vector)